# Schweizerdeutsche Dialekt-Analyse

**Forschungsfrage:** Wie unterscheidet sich der Wortschatz der sieben Schweizerdeutschen Dialektregionen, und können diese Unterschiede zur automatischen Dialekterkennung genutzt werden?

## Schritt 1.1 – Daten laden & explorieren

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

df = pd.read_csv('Data/test.tsv', sep='\t')

print(f'Shape: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten')
print(f'Spalten: {list(df.columns)}')
df.head()

### Verteilung der Dialektregionen

In [ ]:
region_counts = df['dialect_region'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(region_counts.index, region_counts.values, color='#4C72B0')
ax.bar_label(bars, fmt='{:,.0f}', padding=5)
ax.set_xlabel('Anzahl Aufnahmen')
ax.set_title('Anzahl Aufnahmen pro Dialektregion')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

### Verteilung der Aufnahmedauer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm der Dauer
axes[0].hist(df['duration'], bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[0].axvline(df['duration'].median(), color='red', linestyle='--', label=f"Median: {df['duration'].median():.1f}s")
axes[0].set_xlabel('Dauer (Sekunden)')
axes[0].set_ylabel('Anzahl')
axes[0].set_title('Verteilung der Aufnahmedauer')
axes[0].legend()

# Boxplot pro Region
df.boxplot(column='duration', by='dialect_region', ax=axes[1], vert=True, patch_artist=True,
           boxprops=dict(facecolor='#4C72B0', alpha=0.6))
axes[1].set_xlabel('Dialektregion')
axes[1].set_ylabel('Dauer (Sekunden)')
axes[1].set_title('Aufnahmedauer pro Region')
axes[1].tick_params(axis='x', rotation=45)
fig.suptitle('')

plt.tight_layout()
plt.show()

print(df['duration'].describe().to_string())

### Geschlechterverteilung

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie Chart – Geschlecht gesamt
gender_counts = df['gender'].value_counts()
colors_gender = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
axes[0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=colors_gender[:len(gender_counts)], startangle=90)
axes[0].set_title('Geschlechterverteilung (gesamt)')

# Stacked Bar – Geschlecht pro Region
gender_region = pd.crosstab(df['dialect_region'], df['gender'], normalize='index') * 100
gender_region.plot(kind='barh', stacked=True, ax=axes[1], color=colors_gender[:len(gender_region.columns)])
axes[1].set_xlabel('Anteil (%)')
axes[1].set_title('Geschlechterverteilung pro Dialektregion')
axes[1].legend(title='Geschlecht', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

### Altersverteilung

In [ ]:
# Alter in sinnvolle Reihenfolge bringen
age_order = ['teens', 'twenties', 'thirties', 'fourties', 'fifties', 'sixties', 'seventies', 'eighties', 'nineties']
age_labels = [a for a in age_order if a in df['age'].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Balkendiagramm – Alter gesamt
age_counts = df['age'].value_counts().reindex(age_labels)
bars = axes[0].bar(age_counts.index, age_counts.values, color='#55A868', edgecolor='white')
axes[0].bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=9)
axes[0].set_xlabel('Altersgruppe')
axes[0].set_ylabel('Anzahl')
axes[0].set_title('Altersverteilung (gesamt)')
axes[0].tick_params(axis='x', rotation=45)

# Stacked Bar – Alter pro Region
age_region = pd.crosstab(df['dialect_region'], df['age'], normalize='index') * 100
age_region = age_region.reindex(columns=age_labels)
age_region.plot(kind='barh', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_xlabel('Anteil (%)')
axes[1].set_title('Altersverteilung pro Dialektregion')
axes[1].legend(title='Alter', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

### Kantone & Satzquellen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top-15 Kantone
canton_counts = df['canton'].value_counts().head(15).sort_values()
bars = axes[0].barh(canton_counts.index, canton_counts.values, color='#DD8452')
axes[0].bar_label(bars, fmt='{:,.0f}', padding=5)
axes[0].set_xlabel('Anzahl Aufnahmen')
axes[0].set_title('Top-15 Kantone nach Aufnahmeanzahl')

# Satzquellen
source_counts = df['sentence_source'].value_counts()
axes[1].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3'], startangle=90)
axes[1].set_title('Verteilung der Satzquellen')

plt.tight_layout()
plt.show()

## Schritt 1.3 – Filtern (Duration 2–15 Sekunden)

In [ ]:
before = len(df)
df_filtered = df[(df['duration'] >= 2.0) & (df['duration'] <= 15.0)].copy()
after = len(df_filtered)
removed = before - after

print(f'Vorher:    {before:,} Zeilen')
print(f'Nachher:   {after:,} Zeilen')
print(f'Entfernt:  {removed:,} Zeilen ({removed/before*100:.1f}%)')

# Visualisierung: Vorher vs. Nachher
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vorher
axes[0].hist(df['duration'], bins=50, color='#C44E52', edgecolor='white', alpha=0.7, label='Entfernt')
axes[0].hist(df_filtered['duration'], bins=50, color='#4C72B0', edgecolor='white', alpha=0.8, label='Behalten')
axes[0].axvline(2.0, color='black', linestyle='--', linewidth=1.5, label='Filter: 2s')
axes[0].axvline(15.0, color='black', linestyle=':', linewidth=1.5, label='Filter: 15s')
axes[0].set_xlabel('Dauer (Sekunden)')
axes[0].set_ylabel('Anzahl')
axes[0].set_title('Filterung der Aufnahmedauer')
axes[0].legend()

# Verbleibende Aufnahmen pro Region
region_before = df['dialect_region'].value_counts().sort_index()
region_after = df_filtered['dialect_region'].value_counts().sort_index()
x = range(len(region_before))
w = 0.35
axes[1].barh([i + w for i in x], region_before.values, height=w, color='#C44E52', alpha=0.5, label='Vorher')
axes[1].barh(x, region_after.values, height=w, color='#4C72B0', label='Nachher')
axes[1].set_yticks([i + w/2 for i in x])
axes[1].set_yticklabels(region_before.index)
axes[1].set_xlabel('Anzahl Aufnahmen')
axes[1].set_title('Aufnahmen pro Region: Vorher vs. Nachher')
axes[1].legend()

plt.tight_layout()
plt.show()

## Schritt 1.4 – Sample ziehen (200 pro Region)

In [ ]:
# Stratifiziertes Sample: 200 pro Region
samples = []
for region, group in df_filtered.groupby('dialect_region'):
    samples.append(group.sample(n=200, random_state=42))
sample = pd.concat(samples).reset_index(drop=True)

print(f'Sample-Grösse: {len(sample):,} Zeilen')
print(f'Regionen: {sample["dialect_region"].nunique()}')
print()

# Kontrolle
counts = sample['dialect_region'].value_counts().sort_index()
print('=== Aufnahmen pro Region ===')
print(counts.to_string())

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Region
bars = axes[0].barh(counts.index, counts.values, color='#4C72B0')
axes[0].bar_label(bars, padding=5)
axes[0].set_xlabel('Anzahl')
axes[0].set_title('Sample: Aufnahmen pro Region')
axes[0].set_xlim(0, 250)

# Geschlecht im Sample
sample_gender = sample['gender'].value_counts()
axes[1].pie(sample_gender.values, labels=sample_gender.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52'], startangle=90)
axes[1].set_title('Sample: Geschlechterverteilung')

# Alter im Sample
age_order = ['teens', 'twenties', 'thirties', 'fourties', 'fifties', 'sixties', 'seventies', 'eighties', 'nineties']
age_labels = [a for a in age_order if a in sample['age'].unique()]
sample_age = sample['age'].value_counts().reindex(age_labels)
bars2 = axes[2].bar(sample_age.index, sample_age.values, color='#55A868', edgecolor='white')
axes[2].bar_label(bars2, padding=3, fontsize=9)
axes[2].set_xlabel('Altersgruppe')
axes[2].set_ylabel('Anzahl')
axes[2].set_title('Sample: Altersverteilung')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Speichern
sample.to_csv('Data/sample.tsv', sep='\t', index=False)
print(f'\n✅ Sample gespeichert als Data/sample.tsv')

## Schritt 1.5 – Audiodateien prüfen

In [ ]:
import os
import librosa
import librosa.display
import numpy as np

# Ersten Pfad aus dem Sample nehmen
test_path = os.path.join('Data', 'clips__test', sample.iloc[0]['path'])
print(f'Pfad: {test_path}')
print(f'Existiert: {os.path.exists(test_path)}')

if os.path.exists(test_path):
    # Audio laden
    y, sr = librosa.load(test_path, sr=16000)
    duration = librosa.get_duration(y=y, sr=sr)
    print(f'Sample Rate: {sr} Hz')
    print(f'Länge: {duration:.2f} Sekunden')
    print(f'Samples: {len(y):,}')
    
    # Waveform + Mel-Spektrogramm visualisieren
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    
    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#4C72B0')
    axes[0].set_title(f'Waveform – {sample.iloc[0]["dialect_region"]} ({duration:.1f}s)')
    axes[0].set_xlabel('Zeit (s)')
    axes[0].set_ylabel('Amplitude')
    
    # Mel-Spektrogramm
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], cmap='magma')
    axes[1].set_title('Mel-Spektrogramm')
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\n✅ Audiodatei erfolgreich geladen und visualisiert')
else:
    print('❌ Datei nicht gefunden! Pfadstruktur prüfen.')